In [1]:
import sys
import os
sys.path.append(os.path.abspath(".."))

from app.data_providers import provide_dataframe, DataFrameRequest
from processing.fixes import fix_score_column, fix_score_columns_for_all_games
import pandas as pd
import sys

In [2]:
request = DataFrameRequest(
    apply_preprocessing = True,
    add_computed = True,
    filter_pre_encoding_columns = False,
    encode_for_model = False,
    filter_top_players=True,
    filter_clean= True
)

# pipeline execution
df = provide_dataframe(request)


Repo card metadata block was not found. Setting CardData to empty.


# Train test split

In [4]:
df = df.reset_index(drop=True)

In [6]:
df = df.sort_values("GAME_DATE")

In [7]:
games = df[["GAME_ID_x", "GAME_DATE"]].drop_duplicates().sort_values("GAME_DATE")

# Determine split 80/20
split_idx = int(len(games) * 0.8)

train_games = games.iloc[:split_idx]["GAME_ID_x"]
test_games = games.iloc[split_idx:]["GAME_ID_x"]

# data split
train_df = df[df["GAME_ID_x"].isin(train_games)]
test_df = df[df["GAME_ID_x"].isin(test_games)]

print(f"Train size: {len(train_df)}")
print(f"Test size: {len(test_df)}")

Train size: 433221
Test size: 104688


In [10]:
#train_counts = train_df["PLAYER_ID"].value_counts(normalize=True)
#test_counts = test_df["PLAYER_ID"].value_counts(normalize=True)
train_counts = train_df["PLAYER_ID"].value_counts(normalize=False)
test_counts = test_df["PLAYER_ID"].value_counts(normalize=False)

comparison = pd.concat([train_counts, test_counts], axis=1)
comparison.columns = ["train_share", "test_share"]
comparison = comparison.fillna(0)

print(comparison.sort_values("test_share", ascending=False).head(10))

           train_share  test_share
PLAYER_ID                         
203507.0          9974     13234.0
1629029.0         1775     11413.0
2544.0           40468     10351.0
201935.0         21872     10220.0
201939.0         17471      9155.0
203076.0         11769      8866.0
203999.0          4822      8818.0
201566.0         24029      8706.0
201142.0         26409      7830.0
202695.0         10109      6875.0


In [12]:
print(comparison['test_share']/(comparison['test_share'] + comparison['train_share']))

PLAYER_ID
977.0        0.000000
2544.0       0.203684
1717.0       0.000000
1495.0       0.000000
2548.0       0.000000
708.0        0.000000
201142.0     0.228687
2225.0       0.000000
2200.0       0.000000
201566.0     0.265954
201935.0     0.318459
101108.0     0.229493
1938.0       0.000000
201939.0     0.343837
203076.0     0.429658
202695.0     0.404793
203507.0     0.570234
203110.0     0.325251
203999.0     0.646481
1629029.0    0.865408
dtype: float64


# Save the files

In [13]:
df.to_parquet('../data/processed/processed_20_players.parquet')
train_df.to_parquet('../data/processed/processed_20_players_train.parquet')
test_df.to_parquet('../data/processed/processed_20_players_test.parquet')